Nettoyage et imputation

Ce notebook :
1. Charge les données brutes depuis S3 (awswrangler)
2. Détecte et documente les valeurs manquantes
3. Compare deux stratégies d'imputation (SimpleImputer médiane vs KNNImputer)
4. Encode les variables catégorielles avec OneHotEncoder
5. Crée la colonne cible binaire `Churn_flag`
6. Produit un tableau comparatif avant/après (nulls, dtypes)
7. Sauvegarde `telco_clean.csv` sur S3

In [37]:
import awswrangler as wr
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

In [38]:
path_raw = "s3://telco-raw-611284995507-eu-north-1-an/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = wr.s3.read_csv(path_raw)
print(f"Shape initiale : {df.shape}")
df.head()

Shape initiale : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Snapshot AVANT nettoyage (nulls + dtypes)

On capture l'état du DataFrame avant toute transformation, pour pouvoir produire le tableau comparatif avant/après demandé.

In [39]:
snapshot_avant = pd.DataFrame({
    "dtype_avant": df.dtypes.astype(str),
    "nulls_avant": df.isnull().sum()
})
snapshot_avant

,dtype_avant,nulls_avant
customerID,object,0
gender,object,0
SeniorCitizen,int64,0
Partner,object,0
Dependents,object,0
tenure,int64,0
PhoneService,object,0
MultipleLines,object,0
InternetService,object,0
OnlineSecurity,object,0


## 3. Conversion de TotalCharges en numérique


In [40]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
nb_nulls_totalcharges = df["TotalCharges"].isnull().sum()
print(f"Valeurs manquantes détectées dans TotalCharges : {nb_nulls_totalcharges}")

Valeurs manquantes détectées dans TotalCharges : 11


## 4. Comparatif des stratégies d'imputation

On compare `SimpleImputer(strategy="median")` vs `KNNImputer(n_neighbors=5)` sur les 3 colonnes numériques, en conservant les résultats des deux stratégies côte à côte pour pouvoir les comparer avant de choisir laquelle garder.

In [ ]:
cols_numeriques = ["tenure", "MonthlyCharges", "TotalCharges"]

index_nulls = df[df["TotalCharges"].isnull()].index

# --- Stratégie A : SimpleImputer (médiane) ---
imputer_simple = SimpleImputer(strategy="median")
valeurs_simple = imputer_simple.fit_transform(df[cols_numeriques])
df_simple = pd.DataFrame(valeurs_simple, columns=cols_numeriques, index=df.index)

# --- Stratégie B : KNNImputer (retenue comme version finale) ---
imputer_knn = KNNImputer(n_neighbors=5)
valeurs_knn = imputer_knn.fit_transform(df[cols_numeriques])
df_knn = pd.DataFrame(valeurs_knn, columns=cols_numeriques, index=df.index)

# Tableau comparatif des valeurs imputées sur les lignes qui étaient manquantes
comparatif_imputation = pd.DataFrame({
    "TotalCharges_median_imputer": df_simple.loc[index_nulls, "TotalCharges"],
    "TotalCharges_knn_imputer": df_knn.loc[index_nulls, "TotalCharges"],
    "tenure": df.loc[index_nulls, "tenure"],
    "MonthlyCharges": df.loc[index_nulls, "MonthlyCharges"]
})
comparatif_imputation

,TotalCharges_median_imputer,TotalCharges_knn_imputer,tenure,MonthlyCharges
488,1397.475,52.81,0,52.55
753,1397.475,20.25,0,20.25
936,1397.475,80.87,0,80.85
1082,1397.475,25.76,0,25.75
1340,1397.475,55.80,0,56.05
3331,1397.475,19.86,0,19.85
3826,1397.475,25.31,0,25.35
4380,1397.475,19.98,0,20.00
5218,1397.475,19.69,0,19.70
6670,1397.475,73.46,0,73.35


In [42]:
# On retient KNNImputer comme version finale
df[cols_numeriques] = df_knn
print(f"Valeurs manquantes restantes après KNNImputer : {df['TotalCharges'].isnull().sum()}")

Valeurs manquantes restantes après KNNImputer : 0


## 5. Encodage des variables catégorielles (OneHotEncoder)

In [43]:
colonnes_categorielles = ["Contract", "PaymentMethod", "InternetService",
                           "gender", "Partner", "Dependents", "PhoneService",
                           "MultipleLines", "OnlineSecurity", "OnlineBackup",
                           "DeviceProtection", "TechSupport", "StreamingTV",
                           "StreamingMovies", "PaperlessBilling"]

encoder = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore", dtype=int)
encoded_array = encoder.fit_transform(df[colonnes_categorielles])

encoded_cols = encoder.get_feature_names_out(colonnes_categorielles)
df_encoded_cat = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)

for col, cats in zip(colonnes_categorielles, encoder.categories_):
    print(f"{col}: {len(cats)} catégories → {list(cats)}")

Contract: 3 catégories → ['Month-to-month', 'One year', 'Two year']
PaymentMethod: 4 catégories → ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
InternetService: 3 catégories → ['DSL', 'Fiber optic', 'No']
gender: 2 catégories → ['Female', 'Male']
Partner: 2 catégories → ['No', 'Yes']
Dependents: 2 catégories → ['No', 'Yes']
PhoneService: 2 catégories → ['No', 'Yes']
MultipleLines: 3 catégories → ['No', 'No phone service', 'Yes']
OnlineSecurity: 3 catégories → ['No', 'No internet service', 'Yes']
OnlineBackup: 3 catégories → ['No', 'No internet service', 'Yes']
DeviceProtection: 3 catégories → ['No', 'No internet service', 'Yes']
TechSupport: 3 catégories → ['No', 'No internet service', 'Yes']
StreamingTV: 3 catégories → ['No', 'No internet service', 'Yes']
StreamingMovies: 3 catégories → ['No', 'No internet service', 'Yes']
PaperlessBilling: 2 catégories → ['No', 'Yes']


In [ ]:
df_encoded = pd.concat([df["customerID"],df[cols_numeriques], df_encoded_cat], axis=1)

df_encoded["Churn_flag"] = (df["Churn"] == "Yes").astype(int)

print(f"Shape finale : {df_encoded.shape}")
df_encoded.head()

Shape finale : (7043, 31)


,customerID,tenure,MonthlyCharges,TotalCharges,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaperlessBilling_Yes,Churn_flag
0,7590-VHVEG,1.0,29.85,29.85,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
1,5575-GNVDE,34.0,56.95,1889.50,1,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
2,3668-QPYBK,2.0,53.85,108.15,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1
3,7795-CFOCW,45.0,42.30,1840.75,1,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0
4,9237-HQITU,2.0,70.70,151.65,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1


## 6. Tableau comparatif AVANT / APRÈS

In [45]:
resume_avant = pd.DataFrame({
    "metrique": ["Nombre de colonnes", "Nombre de lignes", "Total valeurs manquantes", "Colonnes de type object"],
    "avant": [
        snapshot_avant.shape[0],
        len(df),
        int(snapshot_avant["nulls_avant"].sum()),
        int((snapshot_avant["dtype_avant"] == "object").sum())
    ],
    "apres": [
        df_encoded.shape[1],
        len(df_encoded),
        int(df_encoded.isnull().sum().sum()),
        int((df_encoded.dtypes.astype(str) == "object").sum())
    ]
})
resume_avant

,metrique,avant,apres
0,Nombre de colonnes,21,31
1,Nombre de lignes,7043,7043
2,Total valeurs manquantes,0,0
3,Colonnes de type object,18,1


In [ ]:
df_encoded.dtypes.value_counts()

int32      27
float64     3
object      1
Name: count, dtype: int64

## 7. Sauvegarde du fichier nettoyé sur S3

In [47]:
path_clean = "s3://telco-raw-611284995507-eu-north-1-an/raw/telco_clean.csv"
wr.s3.to_csv(df_encoded, path_clean, index=False)

print(f"Fichier écrit sur : {path_clean}")
print(f"Shape finale : {df_encoded.shape}")
print(f"Nulls restants au total : {df_encoded.isnull().sum().sum()}")

Fichier écrit sur : s3://telco-raw-611284995507-eu-north-1-an/raw/telco_clean.csv
Shape finale : (7043, 31)
Nulls restants au total : 0
